In [3]:
import json
import math
import os

import requests
from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv()
load_dotenv(dotenv_path)
API_KEY = os.getenv("API_KEY")

BASE_URL = "https://api.um.warszawa.pl/api/action/"

Get bus locations:

In [ ]:
endpoint = 'busestrams_get'
BUS_LOC_RESOURCE_ID = 'f2e5503e-927d-4ad3-9500-4ab9e55deb59'
BUS_LOC_URL = BASE_URL + endpoint
type = 1 # 1 dla autobusu, 2 tramwaj

params = {
    'resource_id': BUS_LOC_RESOURCE_ID,
    'apikey': API_KEY,
    'type': type,
    # 'line': 523
}

try:
    res = requests.post(url=BUS_LOC_URL, params=params)
    res.raise_for_status()
    data = res.json()
    print(data)
except Exception as e:
    print(e)

danes = data['result']
for dane in danes:
    print(f"Lat: {dane['Lat']}")
    print(f"Lon: {dane['Lon']}")
    print(f"Time: {dane['Time']}")
    print(f"Lines: {dane['Lines']}")
    print(f"Brigade: {dane['Brigade']}")
    print(f"VehicleNumber: {dane['VehicleNumber']}")
    print(10*'-')

Get przystanki:

In [ ]:
endpoint = "dbtimetable_get"
PRZYSTANKI_URL = BASE_URL + endpoint
PRZYSTANKI_ID = 'ab75c33d-3a26-4342-b36a-6e5fef0a3ac3'
params = {
    'id': PRZYSTANKI_ID,
    'apikey': API_KEY,
    #'name': 'Centrum'
}

try:
    res = requests.get(url=PRZYSTANKI_URL, params=params)
    res.raise_for_status()
    data = res.json()
    print(data)
except Exception as e:
    print(e)

Get linie na przystanku:

In [ ]:
endpoint = "dbtimetable_get"
ROZKLAD_ID = "88cd555f-6f31-43ca-9de4-66c479ad5942"
ROZKLAD_URL = BASE_URL + endpoint

params = {
    'id': ROZKLAD_ID,
    'apikey': API_KEY,
    'busstopId': '7009',
    'busstopNr': '01',
    'line': 523
}

try:
    res = requests.get(url=ROZKLAD_URL, params=params)
    res.raise_for_status()
    data = res.json()
except Exception as e:
    print(f"Błąd: {e}")
print(data)

Rozkład jazdy czasu linii na przystanku

In [ ]:
endpoint = "dbtimetable_get"

ROZKLAD_ID = 'e923fa0e-d96c-43f9-ae6e-60518c9f3238'
URL = BASE_URL + endpoint

params = {
    'id': ROZKLAD_ID,
    'apikey': API_KEY,
    'busstopId': 5030,
    'busstopNr': '02',
    'line': 523
}

try:
    res = requests.get(url=URL, params=params)
    res.raise_for_status()
    data = res.json()

    with open("rozklad523.json", 'w', encoding='utf-8') as f:
        json.dump(data['result'], f, ensure_ascii=False, indent=4)
except Exception as e:
    print(f"Błąd: {e}")

Lista przystanków na trasach linii:

In [ ]:
endpoint = "public_transport_routes"

URL = BASE_URL + endpoint

params = {
    'apikey': API_KEY,
}

try:
    res = requests.get(url=URL, params=params)
    res.raise_for_status()
    data = res.json()
    print(data['result']['523'])
    with open("trasa523.json", 'w', encoding='utf-8') as f:
        json.dump(data['result']['523'], f, ensure_ascii=False, indent=4)
except Exception as e:
    print(f"Błąd: {e}")


## Od teraz należy wykonywać kod po kolei
### 1. Przetwarzanie rozkładu jazdy brygad
Zmiana rozkładu jazdy na normalniejszą formę:

In [ ]:
endpoint = "dbtimetable_get"

ROZKLAD_ID = 'e923fa0e-d96c-43f9-ae6e-60518c9f3238'
URL = BASE_URL + endpoint

busstopId = 2130
busstopNr = '01'
line = 523
params = {
    'id': ROZKLAD_ID,
    'apikey': API_KEY,
    'busstopId': busstopId,
    'busstopNr': busstopNr,
    'line': line
}

try:
    res = requests.get(url=URL, params=params)
    res.raise_for_status()
    data = res.json()

    rozklad_przystanku = {'przystanek_id': f"{busstopId}_{busstopNr}",
                          'line': line,
                          'rozklad': []
                          }
    kursy = data['result']
    for kurs in kursy:
        stop_info = dict()

        for kvp in kurs:
            if kvp['key'] == 'brygada':
                stop_info['brygada'] = kvp['value']
            if kvp['key'] == 'trasa':
                stop_info['trasa'] = kvp['value']
            if kvp['key'] == 'czas':
                stop_info['czas'] = kvp['value']
        
        rozklad_przystanku['rozklad'].append(stop_info)

    with open("rozklad523_dobry.json", 'w', encoding='utf-8') as f:
        json.dump(rozklad_przystanku, f, ensure_ascii=False, indent=4)
except Exception as e:
    print(f"Błąd: {e}")

Obrócenie rozkładu czasu do postaci (w pseudo jsonie)
{
    linia,
    brygada: [
        {info_o_przystanku, czas, trasa}
    ]
}

In [ ]:
with open('rozklad523_dobry.json', 'r') as f:
    data = json.load(f)

    przystanek_id = data['przystanek_id']
    linia = data['line']

    final_json = {'linia': linia}

    for stop in data['rozklad']:
        nr_brygady = stop['brygada']
        if nr_brygady not in final_json:
            final_json[nr_brygady] = []

        final_json[nr_brygady].append(
            {'przystanek_id': przystanek_id,
             'czas': stop['czas'],
             'trasa': stop['trasa']
             }
            )
    
    for brygada in final_json:
        if brygada == 'linia':
            continue
        final_json[brygada].sort(key=lambda x: x['czas'])
        
    with open('rozklad523_obrocony.json', 'w') as out:
        json.dump(final_json, out, ensure_ascii=False, indent=4)

Pobranie i przetworzenie jsona z trasami linii:

In [ ]:
def stworz_trase_linii(linia: int, api_key: str):
    BASE_URL = "https://api.um.warszawa.pl/api/action/"
    endpoint = "public_transport_routes"
    URL = BASE_URL + endpoint

    params = {
        'apikey': API_KEY,
    }

    try:
        res = requests.get(url=URL, params=params)
        res.raise_for_status()
        data = res.json()
        
    except Exception as e:
        print(f"Błąd API przy pobieraniu trasy linii {linia}: {e}")
        return 1

    trasa_linii = data['result'][str(linia)]
    dobra_trasa = {'linia': linia}

    for trasa in trasa_linii:
        dobra_trasa[trasa] = dict()
        for nr_przystanku in trasa_linii[trasa]:
            id_przystanku = trasa_linii[trasa][nr_przystanku]['nr_zespolu']
            nmr_zespolu = trasa_linii[trasa][nr_przystanku]['nr_przystanku']
            odl = trasa_linii[trasa][nr_przystanku]['odleglosc']
            dobra_trasa[trasa][nr_przystanku] = {
                'przystanek_id': f"{id_przystanku}_{nmr_zespolu}",
                'odleglosc': odl
            }

    with open(f"data/trasa_{linia}.json", 'w', encoding='utf-8') as f:
        json.dump(dobra_trasa, f, ensure_ascii=False, indent=4)

    return 0

stworz_trase_linii('523', API_KEY)

Teraz wrzucę do jednego jsona rozkład czasowy wszystkich przystanków na linii, żeby każda brygada miała swój rozkład jazdy. Stworzę jedną funkcję, która pobiera i przetwarza danej dałej linii.

In [ ]:
def stworz_rozklad_linii(linia: int, api_key: str):
    BASE_URL = "https://api.um.warszawa.pl/api/action/"
    endpoint = "dbtimetable_get"
    URL = BASE_URL + endpoint
    ID_ENDPOINT_ROZKLADOW = 'e923fa0e-d96c-43f9-ae6e-60518c9f3238'

    with open(f"data/trasa_{linia}.json", "r") as f:
        json_tras = json.load(f)
        final_json = {'linia': linia, 'brygady': dict()}
        
        unikalne_przystanki_id = set()
        for trasa in json_tras:
            if trasa == 'linia':
                continue
            for przystanek in json_tras[trasa].values():
                unikalne_przystanki_id.add(przystanek['przystanek_id'])

        for przystanek_id in unikalne_przystanki_id:

            przystanek_info = przystanek_id.split('_')
            params = {
                'id': ID_ENDPOINT_ROZKLADOW,
                'apikey': api_key,
                'busstopId': przystanek_info[0],
                'busstopNr': przystanek_info[1],
                'line': linia
            }
            try:
                res = requests.get(url=URL, params=params)
                res.raise_for_status()
                data = res.json()

            except Exception as e:
                print(f"Błąd API przy tworzeniu rozkładu jazdy linii {linia}: {e}")
                return 1

            rozklad_przystanku = {'przystanek_id': przystanek_id,
                                'line': linia,
                                'rozklad': []
                                }
            
            kursy = data['result']
            if not isinstance(kursy, list):
                print(f"Pominęto przystanek {przystanek_id} (brak rozkładu)")
                continue

            for kurs in kursy:
                stop_info = dict()

                for kvp in kurs:
                    if kvp['key'] == 'brygada':
                        stop_info['brygada'] = kvp['value']
                    if kvp['key'] == 'trasa':
                        stop_info['trasa'] = kvp['value']
                    if kvp['key'] == 'czas':
                        stop_info['czas'] = kvp['value']
                
                rozklad_przystanku['rozklad'].append(stop_info)

            for stop in rozklad_przystanku['rozklad']:
                nr_brygady = stop['brygada']
                if nr_brygady not in final_json['brygady']:
                    final_json['brygady'][nr_brygady] = []

                final_json['brygady'][nr_brygady].append(
                    {'przystanek_id': przystanek_id,
                    'czas': stop['czas'],
                    'trasa': stop['trasa']
                    }
                    )
                
    for brygada in final_json['brygady']:
        final_json['brygady'][brygada].sort(key=lambda x: x['czas'])
                
    with open(f'data/rozklad_{linia}.json', 'w', encoding="utf-8") as out:
        json.dump(final_json, out, ensure_ascii=False, indent=4)

    return 0
            

stworz_rozklad_linii(523, API_KEY)

Kolejno, funkcja która tworzy jsona będącego słownikiem id przystanku do jego współrzędnych geograficznych:

In [ ]:
def stworz_baze_polozen_przystankow(api_key: str):
    BASE_URL = "https://api.um.warszawa.pl/api/action/"
    endpoint = "dbtimetable_get"
    PRZYSTANKI_URL = BASE_URL + endpoint
    ID_ENDPOINTU_PRZYSTANKOW = 'ab75c33d-3a26-4342-b36a-6e5fef0a3ac3'
    params = {
        'id': ID_ENDPOINTU_PRZYSTANKOW,
        'apikey': API_KEY,
    }

    try:
        res = requests.get(url=PRZYSTANKI_URL, params=params)
        res.raise_for_status()
        data = res.json()
    except Exception as e:
        print(f"Błąd API przy pobieraniu listy położeń przystanków. {e}")
        return 1

    przystanki = dict()
    for przystanek in data['result']:
        dobry_przystanek = dict()
        for kvp in przystanek['values']:
            if kvp['key'] == 'zespol':
                zespol = kvp['value']
            if kvp['key'] == 'slupek':
                slupek = kvp['value']
            if kvp['key'] == 'szer_geo':
                dobry_przystanek['lat'] = float(kvp['value'])
            if kvp['key'] == 'dlug_geo':
                dobry_przystanek['lon'] = float(kvp['value'])
            if kvp['key'] == 'nazwa_zespolu':
                dobry_przystanek['nazwa_przystanku'] = kvp['value']
            
        przystanek_id = f"{zespol}_{slupek}"
        przystanki[przystanek_id] = dobry_przystanek

    with open('data/przystanki.json', 'w', encoding='utf-8') as f:
        json.dump(przystanki, f, ensure_ascii=False, indent=4)
    
    return 0

stworz_baze_polozen_przystankow(API_KEY)

Funkcja licząca odległość między dwoma punktami we współrzędnych kuli ziemskiej:

In [ ]:
def oblicz_odleglosc(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    R = 6371000.0
    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    lon1_rad = math.radians(lon1)
    lon2_rad = math.radians(lon2)

    delta_lat = abs(lon2_rad - lon1_rad)
    delta_lon = abs(lat2_rad - lat1_rad)

    srednia_lat = (lat1_rad + lat2_rad) / 2.0
    x = delta_lon * math.cos(srednia_lat)
    y = delta_lat
    dystans = R * math.sqrt(x**2 + y**2)

    return round(dystans)

Teraz zajmę się pobieraniem pozycji autobusów:

In [ ]:
endpoint = 'busestrams_get'
BUS_LOC_RESOURCE_ID = 'f2e5503e-927d-4ad3-9500-4ab9e55deb59'
BUS_LOC_URL = BASE_URL + endpoint
type = 1 # 1 dla autobusu, 2 tramwaj

params = {
    'resource_id': BUS_LOC_RESOURCE_ID,
    'apikey': API_KEY,
    'type': type
}

try:
    res = requests.post(url=BUS_LOC_URL, params=params)
    res.raise_for_status()
    data = res.json()
    print(data)
except Exception as e:
    print(e)

wynik = [{'linia': x['Lines'],
          'lat': x['Lat'],
          'lon': x['Lon'],
          'brygada': x['Brigade'],
          'czas': x['Time']
          } for x in data['result'] if (x['Lines'] in ['148'])]

with open('polozenie.json', 'w') as f:
    json.dump(wynik, f, indent=4)